# 03 — Diffusion Model Training

Load the clustered data, build the `DiffusionTransformer1D`, train with `Trainer`, and inspect generated samples mid-run.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import equinox as eqx

from src.data.loader    import load_raw, compute_stats, normalize
from src.data.dataset   import make_windows, train_val_split, numpy_dataloader
from src.models.transformer1d import DiffusionTransformer1D
from src.models.diffusion     import DiffusionProcess
from src.training.train       import Trainer

plt.rcParams['figure.dpi'] = 110
print('JAX devices:', jax.devices())

## 1. Load & prepare data

In [ ]:
df = load_raw('../data/power.pk')

clusters_df = pd.read_csv('../data/clusters.csv')
cluster_labels = clusters_df['cluster_id'].values            # (N,)
N_CLUSTERS = int(cluster_labels.max()) + 1

print(f'{df.shape[1]} meters, {N_CLUSTERS} clusters')

# Build timestamps (needed for day-type conditioning)
# Assume 15-min resolution starting at an arbitrary Monday date
timestamps = pd.date_range('2010-01-04', periods=df.shape[0], freq='15min')  # 2010-01-04 is a Monday
if isinstance(df.index, pd.DatetimeIndex):
    timestamps = df.index

stats = compute_stats(df, cluster_labels)
df_norm = normalize(df, stats, cluster_labels)

In [ ]:
xs, cs, mid = make_windows(df_norm, cluster_labels, timestamps)
print(f'Windows: xs={xs.shape}  cs={cs.shape}  mid={mid.shape}')

x_train, c_train, x_val, c_val = train_val_split(xs, cs, mid, n_meters=df.shape[1])
print(f'Train: {x_train.shape[0]}  Val: {x_val.shape[0]}')

## 2. Model & diffusion setup

In [ ]:
BATCH_SIZE   = 64
N_EPOCHS     = 50
LR           = 1e-3
WARMUP_STEPS = 500
GUIDANCE_SCALE = 1.5

key = jax.random.PRNGKey(0)

model = DiffusionTransformer1D(
    seq_len    = 96,
    d_model    = 128,
    n_heads    = 4,
    n_layers   = 4,
    d_ff       = 256,
    n_clusters = N_CLUSTERS,
    n_day_types= 2,
    key        = key,
)

diffusion = DiffusionProcess(T=1000, freq_loss_weight=0.05)
trainer   = Trainer(model, diffusion, lr=LR, warmup_steps=WARMUP_STEPS)

n_params = sum(x.size for x in jax.tree_util.tree_leaves(eqx.filter(model, eqx.is_array)))
print(f'Model parameters: {n_params:,}')

## 3. Training

In [ ]:
train_loader = numpy_dataloader(x_train, c_train, batch_size=BATCH_SIZE, shuffle=True,  rng=np.random.default_rng(1))
val_loader   = numpy_dataloader(x_val,   c_val,   batch_size=BATCH_SIZE, shuffle=False, rng=np.random.default_rng(2))

train_losses, val_losses = trainer.fit(
    train_loader = train_loader,
    val_loader   = val_loader,
    n_epochs     = N_EPOCHS,
    steps_per_epoch  = max(1, len(x_train) // BATCH_SIZE),
    val_steps        = max(1, len(x_val)   // BATCH_SIZE),
    checkpoint_dir   = '../checkpoints',
    print_every_epochs = 5,
)
print('Training complete.')

## 4. Loss curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label='Train', color='steelblue')
ax.plot(val_losses,   label='Val',   color='coral')
ax.set_xlabel('Epoch'); ax.set_ylabel('Noise-prediction loss')
ax.set_title('Diffusion training loss')
ax.legend(); plt.tight_layout(); plt.show()

## 5. Quick sample inspection (DDIM 50 steps)

In [ ]:
hours = np.arange(96) / 4
n_show = 4

fig, axes = plt.subplots(N_CLUSTERS, 2, figsize=(12, 3 * N_CLUSTERS))

for cid in range(N_CLUSTERS):
    for dt, day_label in enumerate(['Weekday', 'Weekend']):
        c_batch = jnp.array([[cid, dt]] * n_show)
        gen_key = jax.random.PRNGKey(cid * 10 + dt)
        samples = diffusion.ddim_sample(
            trainer.model, c_batch,
            seq_len=96, batch_size=n_show,
            key=gen_key, n_steps=50, guidance_scale=GUIDANCE_SCALE
        )   # (n_show, 96)
        samples = np.array(samples)
        ax = axes[cid, dt]
        for i in range(n_show):
            ax.plot(hours, samples[i], alpha=0.7, linewidth=1)
        ax.set_title(f'Cluster {cid} — {day_label}')
        ax.set_xlabel('Hour')

plt.suptitle('DDIM samples (normalised space)', fontsize=12)
plt.tight_layout(); plt.show()

## 6. Save final checkpoint

In [ ]:
trainer.save('../checkpoints/best_model.pkl')
print('Checkpoint saved.')